In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [ ]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("test1")

In [ ]:
from typing import List, Optional, Dict, Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_teddynote.graphs import visualize_graph


# ==================== 속성/메타데이터 정의 ====================
class QueryAnalysis(BaseModel):
    """쿼리 분석 결과"""

    is_ambiguous: bool = Field(description="쿼리가 모호한지 여부")
    ambiguity_reasons: List[str] = Field(description="모호성 이유 목록")
    refined_query: str = Field(description="정제된 쿼리")
    search_keywords: List[str] = Field(description="추출된 검색 키워드 목록")
    domain: Optional[str] = Field(description="연구 도메인")
    methodology: Optional[str] = Field(description="연구 방법론")
    time_period: Optional[str] = Field(description="연구 기간")


class Paper(BaseModel):
    """논문 정보"""

    title: str = Field(description="논문 제목")
    authors: str = Field(description="저자")
    abstract: str = Field(description="초록")
    source: str = Field(description="출처")
    published_date: Optional[str] = Field(description="출판일")
    url: Optional[str] = Field(description="논문 URL")
    content: Optional[str] = Field(description="논문 전체 내용")


class Limitation(BaseModel):
    """논문의 한계점"""

    paper_title: str = Field(description="논문 제목")
    limitation_text: str = Field(description="한계점 원문")
    category: str = Field(
        description="한계 카테고리: 데이터 의존도, 강건성, 확장성, 구조적 맹점, 실사용성"
    )
    severity: str = Field(description="심각도: high, medium, low")


class ResearchGap(BaseModel):
    """연구 GAP"""

    category: str = Field(description="GAP 카테고리")
    description: str = Field(description="GAP 설명")
    supporting_papers: List[str] = Field(
        description="이 GAP을 뒷받침하는 논문 제목 목록"
    )
    frequency: int = Field(description="이 GAP이 등장한 빈도")
    proposed_research_topic: str = Field(description="제안된 연구 주제")
    rationale: str = Field(description="연구 주제 제안 근거")


# ==================== 상태 정의 ====================


class ResearchGapAnalysisState(TypedDict):
    # 사용자 입력
    original_query: str
    user_feedback: Optional[str]

    # Query Analysis Agent 출력
    query_analysis: Optional[QueryAnalysis]
    refinement_count: int

    # Research Intelligence Agent 출력
    papers: List[Paper]
    web_contents: List[str]

    # GAP Inference Agent 출력
    limitations: List[Limitation]
    research_gaps: List[ResearchGap]

    # 제어 플래그
    need_user_feedback: bool
    max_refinement: int
    current_step: str

In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_teddynote.tools.tavily import TavilySearch
from langchain_community.retrievers import ArxivRetriever
from langchain_core.runnables import RunnableConfig
from langchain_teddynote.messages import random_uuid, invoke_graph
import operator
from typing import Annotated

In [ ]:
# ==================== Query Analysis Agent ====================

query_analysis_prompt = """당신은 연구 쿼리를 분석하고 정제하는 전문가입니다.

사용자의 연구 질문을 다음 기준으로 분석하세요:
1. 연구 도메인이 명확한가?
2. 대상 데이터나 방법론이 구체적인가?
3. 연구 기간이나 범위가 명시되어 있는가?

사용자 질문: {query}

{feedback}

다음을 수행하세요:
1. 질문의 모호성을 판단하고 이유를 나열
2. 논문 검색에 적합하도록 쿼리를 정제 (동의어 확장, 방법론 키워드 추가, 필터링 조건)
3. 핵심 검색 키워드 3-7개를 추출 (고정값이 아님, 쿼리 복잡도에 따라 동적 결정)
4. 연구 도메인, 방법론, 기간 등 메타정보 추출
"""


def query_analysis_agent(state: ResearchGapAnalysisState):
    """쿼리 분석 및 정제"""
    original_query = state["original_query"]
    user_feedback = state.get("user_feedback", "")
    refinement_count = state.get("refinement_count", 0)

    feedback_text = f"\n사용자 피드백: {user_feedback}" if user_feedback else ""

    structured_llm = llm.with_structured_output(QueryAnalysis)

    system_message = query_analysis_prompt.format(
        query=original_query, feedback=feedback_text
    )

    analysis = structured_llm.invoke(
        [
            SystemMessage(content=system_message),
            HumanMessage(content="쿼리를 분석하고 정제해주세요."),
        ]
    )

    # 모호성 검증: 3회 이상 정제하거나 모호하지 않으면 다음 단계로
    need_feedback = analysis.is_ambiguous and refinement_count < 3

    return {
        "query_analysis": analysis,
        "refinement_count": refinement_count + 1,
        "need_user_feedback": need_feedback,
        "current_step": "query_analysis",
    }


def user_feedback_node(state: ResearchGapAnalysisState):
    """사용자 피드백 수집을 위한 중단점"""
    pass


def should_continue_query_analysis(state: ResearchGapAnalysisState):
    """쿼리 분석 후 다음 단계 결정"""
    if state.get("need_user_feedback", False):
        return "user_feedback"
    return "paper_search"

In [ ]:
# ==================== Research Intelligence Agent ====================


def paper_search_agent(state: ResearchGapAnalysisState):
    """논문 검색 및 매칭"""
    query_analysis = state["query_analysis"]
    search_keywords = query_analysis.search_keywords

    # ArXiv 검색
    arxiv_retriever = ArxivRetriever(
        load_max_docs=5,
        load_all_available_meta=True,
        get_full_documents=True,
    )

    papers = []

    # 각 키워드로 검색
    for keyword in search_keywords[:3]:  # 상위 3개 키워드로 검색
        try:
            results = arxiv_retriever.invoke(keyword)
            for doc in results:
                paper = Paper(
                    title=doc.metadata.get("Title", ""),
                    authors=doc.metadata.get("Authors", ""),
                    abstract=doc.metadata.get("Summary", ""),
                    source=doc.metadata.get("entry_id", ""),
                    published_date=doc.metadata.get("Published", ""),
                    url=doc.metadata.get("entry_id", ""),
                    content=doc.page_content[:10000],  # 컨텐츠 길이 제한
                )
                papers.append(paper)
        except Exception as e:
            print(f"검색 오류 ({keyword}): {e}")

    # 중복 제거 (제목 기준)
    seen_titles = set()
    unique_papers = []
    for paper in papers:
        if paper.title not in seen_titles:
            seen_titles.add(paper.title)
            unique_papers.append(paper)

    return {
        "papers": unique_papers[:10],  # 상위 10개만 유지
        "current_step": "paper_search",
    }


def web_search_agent(state: ResearchGapAnalysisState):
    """웹 검색 - 최신 연구 동향"""
    query_analysis = state["query_analysis"]
    search_keywords = query_analysis.search_keywords

    tavily_search = TavilySearch(max_results=3)

    web_contents = []

    # 상위 2개 키워드로 웹 검색
    for keyword in search_keywords[:2]:
        try:
            results = tavily_search.invoke(keyword + " research latest trends")
            for doc in results:
                content = f"[{doc['url']}]\n{doc['content']}"
                web_contents.append(content)
        except Exception as e:
            print(f"웹 검색 오류 ({keyword}): {e}")

    return {"web_contents": web_contents, "current_step": "web_search"}

In [ ]:
# ==================== GAP Inference Agent ====================

limitation_extraction_prompt = """당신은 논문의 한계점을 추출하는 전문가입니다.

다음 논문에서 Limitations, Future Work, Discussion 섹션에 언급된 한계점을 추출하세요.

논문 제목: {title}
논문 내용:
{content}

각 한계점을 다음 카테고리 중 하나로 분류하세요:
- 데이터 의존도: 특정 데이터셋이나 도메인에 대한 의존성
- 강건성: 다양한 조건에서의 안정성 부족
- 확장성: 대규모 적용의 어려움
- 구조적 맹점: 모델이나 방법론의 근본적 한계
- 실사용성: 실제 환경 적용의 어려움

최대 5개의 주요 한계점만 추출하세요.
"""


def extract_limitations(state: ResearchGapAnalysisState):
    """논문에서 한계점 추출"""
    papers = state["papers"]
    web_contents = state.get("web_contents", [])

    limitations = []

    # 각 논문에서 한계점 추출
    for paper in papers[:5]:  # 상위 5개 논문만 분석
        try:

            class LimitationList(BaseModel):
                limitations: List[Limitation]

            structured_llm = llm.with_structured_output(LimitationList)

            prompt = limitation_extraction_prompt.format(
                title=paper.title,
                content=paper.content[:8000] if paper.content else paper.abstract,
            )

            result = structured_llm.invoke(
                [
                    SystemMessage(content=prompt),
                    HumanMessage(content="한계점을 추출해주세요."),
                ]
            )

            limitations.extend(result.limitations)
        except Exception as e:
            print(f"한계점 추출 오류 ({paper.title}): {e}")

    return {"limitations": limitations, "current_step": "extract_limitations"}


gap_inference_prompt = """당신은 연구 GAP을 도출하는 전문가입니다.

다음 한계점들을 분석하여 반복적으로 나타나는 핵심 연구 GAP을 도출하세요.

추출된 한계점들:
{limitations}

분석 기준:
1. 여러 논문에서 반복적으로 등장하는 한계점
2. 사용자의 원래 질문과의 관련성
3. 해결의 시급성과 중요성

카테고리별로 가장 중요한 GAP 3-5개를 도출하고, 각 GAP에 대해:
- 새로운 연구 주제를 제안
- 제안 근거를 명확히 설명

사용자의 원래 질문: {original_query}
"""


def infer_research_gaps(state: ResearchGapAnalysisState):
    """연구 GAP 추론 및 연구 주제 생성"""
    limitations = state["limitations"]
    original_query = state["original_query"]

    # 한계점을 텍스트로 포맷팅
    limitations_text = "\n\n".join(
        [
            f"[{lim.paper_title}] ({lim.category}, {lim.severity})\n{lim.limitation_text}"
            for lim in limitations
        ]
    )

    class ResearchGapList(BaseModel):
        gaps: List[ResearchGap]

    structured_llm = llm.with_structured_output(ResearchGapList)

    prompt = gap_inference_prompt.format(
        limitations=limitations_text, original_query=original_query
    )

    result = structured_llm.invoke(
        [
            SystemMessage(content=prompt),
            HumanMessage(
                content="연구 GAP을 도출하고 새로운 연구 주제를 제안해주세요."
            ),
        ]
    )

    return {"research_gaps": result.gaps, "current_step": "infer_gaps"}

In [ ]:
# ==================== 그래프 구성 ====================


def build_research_gap_graph():
    """연구 GAP 분석 그래프 구성"""

    builder = StateGraph(ResearchGapAnalysisState)

    # 노드 추가
    builder.add_node("query_analysis", query_analysis_agent)
    builder.add_node("user_feedback", user_feedback_node)
    builder.add_node("paper_search", paper_search_agent)
    builder.add_node("web_search", web_search_agent)
    builder.add_node("extract_limitations", extract_limitations)
    builder.add_node("infer_gaps", infer_research_gaps)

    # 엣지 연결
    builder.add_edge(START, "query_analysis")
    builder.add_conditional_edges(
        "query_analysis",
        should_continue_query_analysis,
        ["user_feedback", "paper_search"],
    )
    builder.add_edge("user_feedback", "query_analysis")
    builder.add_edge("paper_search", "web_search")
    builder.add_edge("web_search", "extract_limitations")
    builder.add_edge("extract_limitations", "infer_gaps")
    builder.add_edge("infer_gaps", END)

    # 메모리 및 컴파일
    memory = MemorySaver()
    graph = builder.compile(interrupt_before=["user_feedback"], checkpointer=memory)

    return graph

In [ ]:
# ==================== 실행 ====================

# 그래프 생성
graph = build_research_gap_graph()

# 그래프 시각화
visualize_graph(graph)

# 설정
config = RunnableConfig(
    recursion_limit=50,
    configurable={"thread_id": random_uuid()},
)

# 연구 주제 설정
research_topic = "Modular RAG가 기존의 Naive RAG와 어떤 차이가 있는지와 production level에서 사용하는 이점"

# 초기 입력
initial_state = {
    "original_query": research_topic,
    "user_feedback": None,
    "refinement_count": 0,
    "max_refinement": 3,
    "papers": [],
    "web_contents": [],
    "limitations": [],
    "research_gaps": [],
}

# 그래프 실행
print("=" * 80)
print("연구 GAP 분석 시작")
print("=" * 80)

result = invoke_graph(graph, initial_state, config)